In [1]:
# library
import os
import json
import glob
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import asyncio
import websockets
import time
import pytz
from datetime import datetime, timedelta
from IPython.display import clear_output
import wavio
import sounddevice
import soundfile
import warnings

# specific
import re
from lxml import html
from bs4 import BeautifulSoup, Tag
from selenium import webdriver
from selenium.webdriver.chrome.options import Options as chrome_options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException

In [2]:
# common functions
def header(s, n=0):
    t = f"<<---  {s}  --->>"
    for i in range(n):
        t += "\n"
    print(t)

def view(obj):
    print(type(obj))
    print(obj)

def view_full_df(df):
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 200):
        print(df)

def mkdir(PATH):
    for path in PATH:
        os.makedirs(path, exist_ok=True)

def save_file(data, path):
    if isinstance(data, pd.DataFrame):
        table = pa.Table.from_pandas(data)
        pa.feather.write_feather(table, path)
    elif isinstance(data, np.ndarray):
        np.save(path, data)
    elif isinstance(data, dict):
        with open(path, 'w') as file:
            json.dump(data, file)

def load_file(path):
    root, ext = os.path.splitext(path)
    if ext == '.csv':
        return pd.read_csv(path)
    elif ext == '.xls':
        return pd.read_excel(path, engine='xlrd')
    elif ext == '.npy':
        return np.load(path, allow_pickle=True)
    elif ext == '.fea':
        return pd.read_feather(path)
    elif ext == '.json':
        with open(path, 'r') as file:
            data = json.load(file)
            return data

def halt(sec=5):
    time.sleep(sec)

def get_date_time():
    tz = pytz.timezone('Asia/Ho_Chi_Minh')
    t = datetime.now(tz)
    return int(t.strftime('%Y%m%d')), t.hour * 100 + t.minute

def screen(s):
    clear_output(wait=True)
    print(s)

def record_sound(path):
    fs = 44100
    duration = 5
    recording = sounddevice.rec(int(duration * fs), samplerate=fs, channels=2)
    sounddevice.wait()
    wavio.write(path, recording, fs, sampwidth=3)

def sound_alarm(type, n=3):
    path = f"{home}/data/sound/{type}.wav"
    data, fs = soundfile.read(path, dtype='float32')
    for i in range(n):
        sounddevice.play(data, fs)
        sounddevice.wait()

In [3]:
# function library
def isfloat(s):
    try:
        float(s)
        return True
    except:
        return False

def isint(s):
    try:
        int(s)
        return True
    except:
        return False

def build_DCOL(COL):
    DCOL, n = {}, len(COL)
    for i in range(n):
        DCOL[COL[i]] = i
    return DCOL

def vstack_row(arr, row, pgdn=True):
    result = None
    if arr is None:
        result = np.array([row])
    else:
        result = np.vstack([arr, row]) if pgdn == True else np.vstack([row, arr])
    return result

def get_arr_data(stk, COL=None):
    DCOL = {}
    for i in range(len(COL_PV)):
        DCOL[COL_PV[i]] = i
    data = load_file(f"{home}/data/pv/{stk}.npy")
    arr = []
    if 'date' not in COL:
        arr = [data[:, 0]]
    for col in COL:
        arr.append(data[:, DCOL[col]])
    return arr

def arr_to_df(arr, D):
    df = pd.DataFrame(arr, columns=D.keys())
    for col in D.keys():
        df[col] = df[col].astype(D[col])
    return df

def merge_df(df1, df2, COL=['d', 'date', 'stk']):
    return df1.merge(df2, on=COL, how='left')

def histogram(data, col=None):
    if isinstance(data, pd.DataFrame):
        print(data[col].describe())
        data[col].hist(bins=100)
        plt.show()
    elif isinstance(data, np.ndarray):
        LABEL = ['count', 'min', 'qt25', 'med', 'qt75', 'max']
        VALUE = [np.count_nonzero(~np.isnan(data)), np.nanmin(data), np.nanquantile(data, 0.25), \
        np.nanmedian(data), np.nanquantile(data, 0.75), np.max(data)]
        for i in range(len(LABEL)):
            print(f"{LABEL[i]}  =  {VALUE[i]}")
        plt.hist(data, bins=100)
        plt.show()

def pasteurize_arr(x, y):
    a = [u for u, v in zip(x, y) if not (np.isnan(u) or np.isnan(v))]
    b = [v for u, v in zip(x, y) if not (np.isnan(u) or np.isnan(v))]
    a, b = np.array(a), np.array(b)
    return a, b

def plot_scatter(x, y):
    a, b = pasteurize_arr(x, y)
    plt.scatter(a, b, marker='o', color='blue', label='Scatter Plot')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Scatter Plot with Regression Line')
    model = Ridge(alpha=1.0)
    model.fit(a.reshape(-1, 1), b)
    a_range = np.linspace(min(a), max(b), 100)
    b_range = model.predict(a_range.reshape(-1, 1))
    plt.plot(a_range, b_range, color='red', label='Regression Line')
    plt.legend()
    plt.show()

def plot_ts(df, col, title=None, edge=False):
    df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
    plt.figure(figsize=(8, 4))
    plt.plot(df['date'], df[col], label=col)
    if edge == True:
        for x, y in zip(df['date'], df[col]):
            plt.text(x, y, f'{y:.4f}', fontsize=8)
    plt.xlabel('date')
    plt.ylabel('values')
    plt.legend()
    if title is not None:
        plt.title(title)
    plt.show()

def plot_ts_data(data, col, window, d, title=None):
    df = pd.DataFrame({'date': data[d - window + 1 : d + 1, 0], col: data[d - window + 1 : d + 1, DPV[col]]})
    plot_ts(df, col, title)

In [4]:
# computing functions
def count_nnan(arr):
    return np.count_nonzero(~np.isnan(arr))

def get_chgn(d, stk, window, col='adjcls'):
    data = load_file(f"{home}/data/pv/{stk}.npy")
    data = data[d - window : d + 1, DPV[col]]
    return np.diff(data) / data[: -1]

def chng(b, a, mode=0):
    c = b / a - 1
    if mode == 0:
        c = round(c * 100, 2)
    return c

def get_quantile(seq, value):
    qt = np.nan
    try:
        arr = sorted(seq)
        i = arr.index(value)
        qt = i / len(arr)
    except:
        pass
    return qt

def round_prc(value):
    if np.isnan(value):
        return np.nan
    num = 10 * value
    a, b = int(num), num % 1
    c = 0
    if (0.25 < b) and (b < 0.75):
        c = 0.5
    elif b >= 0.75:
        c = 1
    return (a + c) / 10

def get_reversality(d, stk, window, col='adjcls'):
    data = get_chgn(d, stk, window + 1, col)
    a, b = data[1 :], data[: -1]
    c = a * b
    rate = count_nnan(c) / window
    value = np.nan if rate < 0.8 else np.nansum(c)
    return value

def get_corr(df, x, y, dspl=False):
    value = df[x].corr(df[y])
    if dspl == True:
        print(f"corr_{x}_{y}  =  {value}")
    else:
        return value

In [5]:
# generic functions
def gen_point(point, para):
    return Dfunc[para[0]](point, para)

def glue_point(stat, D):
    stat = np.vstack(stat)
    E = {'date': 'int', 'd': 'int', 'stk': 'str'}
    E = {**E, **D}
    stat = pd.DataFrame(stat, columns=E.keys())
    for col in E.keys():
        stat[col] = stat[col].astype(E[col])
    return stat

def gen_event(axis, para, mode=0):
    if mode == 1:
        point = axis[0]
        stat = gen_point(point, para)
        return stat
    stat = Parallel(n_jobs=50)(delayed(gen_point)(point, para) for point in axis)
    stat = glue_point(stat, para[-1])
    return stat

In [6]:
# detailed functions
def trace_1(stk, para):  # reversality
    date = load_file(f"{home}/data/universe/date.npy")
    n, window = len(date), para[1]
    stat = None
    for d in range(n):
        value = get_reversality(d, stk, window)
        stat = vstack_row(stat, [int(date[d]), int(d), stk, value])
    return stat

def trace_2(stk, para):  # chgn(cls, opn)
    date, data = load_file(f"{home}/data/universe/date.npy"), load_file(f"{home}/data/pv/{stk}.npy")
    n = len(date)
    stat = None
    for d in range(n):
        value = data[d, DPV['close']] / data[d, DPV['open']] - 1.0
        stat = vstack_row(stat, [int(date[d]), int(d), stk, value])
    return stat

def trace_3(stk, para):  # cls[d] / cls[d-1] - 1
    date, data = load_file(f"{home}/data/universe/date.npy"), load_file(f"{home}/data/pv/{stk}.npy")
    n = len(date)
    stat = None
    for d in range(n):
        value = data[d, DPV['adjcls']] / data[d - 1, DPV['adjcls']] - 1.0
        stat = vstack_row(stat, [int(date[d]), int(d), stk, value])
    return stat

def trace_4(stk, para):  # jht
    date, data = load_file(f"{home}/data/universe/date.npy"), load_file(f"{home}/data/pv/{stk}.npy")
    n, window = len(date), para[1]
    stat = None
    for d in range(n):
        value = np.nan
        try:
            arr = [data[i, DPV['adjcls']] / data[d, DPV['adjcls']] - 1.0 for i in range(d + 2, d + 2 + window)]
            rate = count_nnan(arr) / window
            if rate >= 0.8:
                value = np.nanmax(arr)
        except:
            pass
        stat = vstack_row(stat, [int(date[d]), int(d), stk, value])
    return stat

def trace_5(stk, para):  # jht >= price_sell
    date, data = load_file(f"{home}/data/universe/date.npy"), load_file(f"{home}/data/pv/{stk}.npy")
    n, window, rlong, rshort = len(date), para[1], para[2], para[3]
    stat = None
    for d in range(n):
        value = np.nan
        try:
            arr = data[d + 2 : d + 2 + window, DPV['adjcls']]
            rate = count_nnan(arr) / window
            if rate >= 0.8:
                value = np.nanmax(arr) / (1.0 + rshort) / (1.0 - rlong) / data[d, DPV['adjopn']] - 1.0
        except:
            pass
        stat = vstack_row(stat, [int(date[d]), int(d), stk, value])
    return stat

def trace_6(stk, para):  # data_raw, [mode, col, {}]
    date, data = load_file(f"{home}/data/universe/date.npy"), load_file(f"{home}/data/pv/{stk}.npy")
    n, col = len(date), para[1]
    stat = None
    for d in range(n):
        value = data[d, DPV[col]]
        stat = vstack_row(stat, [int(date[d]), int(d), stk, value])
    return stat

def trace_7(stk, para):  # max, [mode, col, window, {}]
    date, data = load_file(f"{home}/data/universe/date.npy"), load_file(f"{home}/data/pv/{stk}.npy")
    n, col, window = len(date), para[1], para[2]
    stat = None
    for d in range(n):
        value = np.nan
        rate = count_nnan(data[d + 2 : d + 2 + window, DPV[col]]) / window
        if rate >= 0.8:
            value = np.nanmax(data[d + 2 : d + 2 + window, DPV[col]])
        stat = vstack_row(stat, [int(date[d]), int(d), stk, value])
    return stat

Dfunc = {int(name.split('_')[1]): func for name, func in globals().items() if callable(func) and name.startswith("trace_")}

In [7]:
# constants
home = r'D:\Notebook\VNS'
path_vcbs, path_hsx, path_cafef = f"{home}/data/vcbs", f"{home}/data/hsx", f"{home}/data/cafef"
path_intra = f"{home}/data/intraday"

COL_vcbs = ['date', 'adj', 'floor', 'ceil', 'opn', 'cls', 'low', 'high', 'nsh', 'Fb', 'Fs', 'Fh']
COL_hsx = ['date', 'adj', 'floor', 'ceil', 'opn', 'cls', 'low', 'high', 'avr', 'diff', 'chng', 'nsh', 'vol']
COL_cafef = ['date', 'ret', 'opn', 'cls', 'adj', 'high', 'low', 'vol', 'vol2']
COL_hist = ['date', 'floor', 'ceil', 'opn', 'cls', 'low', 'high', 'nsh', 'vol', 'adj']
COL_pv = ['date', 'floor', 'ceil', 'opn', 'cls', 'low', 'high', 'adj', 'aopn', 'acls', 'alow', 'ahigh', 'aret', 'nsh', 'vol', 'Fb', 'Fs', 'Fh']
COL_vcbsr = ['date', 'exch', 'stk', 'ceil', 'floor', 'adj', 'b3', 'nshb3', 'b2', 'nshb2', 'b1', 'nshb1', 'chng2', 'cls', 'nshcls', \
                's1', 'nshs1', 's2', 'nshs2', 's3', 'nshs3', 'nsh', 'opn', 'high', 'low', 'Fb', 'Fs', 'Fh']
COL_vni = ['date', 'delta', 'aret', 'aopn', 'alow', 'ahigh', 'acls', 'avr', 'nsh', 'vol']


Dvni, Dhsx, Dcafef, Dhist, Dpv = build_DCOL(COL_vni), build_DCOL(COL_hsx), build_DCOL(COL_cafef), build_DCOL(COL_hist), build_DCOL(COL_pv)
Dvcbs, Dvcbsr = build_DCOL(COL_vcbs), build_DCOL(COL_vcbsr)

date_hsx, date_vcbs = 20110504, 20231225
vcbs_const = load_file(f"{path_vcbs}/live/vcbs_const.json")

# vcbs_const = {'hose': ['//*[@id="HOSE_Group"]/a/span', '//*[@id="tab4"]/p', 416], \
#               'hnx': ['//*[@id="HNX_Group"]/a/span', '//*[@id="tab7"]/p', 326], \
#           'vn30': ['//*[@id="HOSE_Group"]/a/span', '//*[@id="tab6"]/p', 32], \
#               'hnx30': ['//*[@id="HNX_Group"]/a/span', '//*[@id="tab9"]/p', 32]}

In [8]:
def clear_folder(path):
    files = glob.glob(os.path.join(path, '*'))
    for file in files:
        if os.path.isfile(file):
            os.remove(file)

def collect_stk():
    def get_exch(EXCH, exch):
        xp1, xp2 = EXCH[exch][0], EXCH[exch][1]
        wait = WebDriverWait(driver, 10)
        parent = wait.until(EC.presence_of_element_located((By.XPATH, xp1)))
        ActionChains(driver).move_to_element(parent).perform()
        child = wait.until(EC.presence_of_element_located((By.XPATH, xp2)))
        child = wait.until(EC.element_to_be_clickable((By.XPATH, xp2)))
        child.click()
        time.sleep(3)
        STK, html = [], driver.find_element(By.XPATH, xp)
        S =  html.get_attribute('innerHTML')
        DATA, soup = None, BeautifulSoup(S, 'html.parser')
        BLOCK = soup.find_all('tr')
        for block in BLOCK:
            if block.has_attr('name'):
                stk = block['name']
                if stk != 's_8_s':
                    STK.append(stk)
        return STK
    STK, EXCH = [], {'hose': ['//*[@id="HOSE_Group"]/a/span', '//*[@id="tab4"]/p'], 'hnx': ['//*[@id="HNX_Group"]/a/span', '//*[@id="tab7"]/p']}
    url, xp = "https://priceboard.vcbs.com.vn/Priceboard#", '//*[@id="priceboardContentTableBody"]'
    opt = chrome_options()
    opt.add_argument("--headless")
    driver = webdriver.Chrome(opt)
    driver.get(url)
    for exch in EXCH.keys():
        STK = STK + get_exch(EXCH, exch)
    STK = list(set(STK))
    return STK

# def collect(path, s):
#     def cl1(t):
#         if t[: 2] == "S#":
#             return t[2 :]
#         else:
#             return None
#     L = s.split('|')
#     n = len(L)
#     if n >= 90:
#         try:
#             stk, ret, nsh, vol = cl1(L[0]), float(L[52]), float(L[53]), round(float(L[54]) / 1e9, 2)
#             trdg, low, high, adj, avr = float(L[41]), float(L[45]), float(L[43]), float(L[60]), float(L[46])
#             row = [adj, trdg, low, high, avr, ret, nsh, vol]
#             path_stk = f"{path}/{stk}.npy"
#             data = None if not os.path.exists(path_stk) else load_file(path_stk)
#             data = vstack_row(data, row)
#             save_file(data, path_stk)
#         except:
#             pass

def read_message(msg, date):
    def sp(s, mode=0):
        t = 0
        if mode == 0:
            try:
                t = float(s)
            except:
                pass
        elif mode == 1:
            try:
                t = int(s)
            except:
                pass
        return t
    L = msg.split('|')
    if len(L) >= 102:
        tm = datetime.now()
        stk = str(L[1][2 :])
        trdg, diff, chng = sp(L[42]) / 1000, sp(L[52]) / 1000, sp(L[53])
        high, low, nstr, ns = sp(L[44]) / 1000, sp(L[46]) / 1000, sp(L[43], 1), sp(L[54], 1)
        ceil, floor, adj = sp(L[59]) / 1000, sp(L[60]) / 1000, sp(L[61]) / 1000
        b1, b2, b3 = sp(L[2]) / 1000, sp(L[4]) / 1000, sp(L[6]) / 10000
        s1, s2, s3 = sp(L[22]) / 1000, sp(L[24]) / 1000, sp(L[26]) / 10000
        nsb1, nsb2, nsb3 = sp(L[3], 1), sp(L[5], 1), sp(L[7], 1)
        nss1, nss2, nss3 = sp(L[23], 1), sp(L[25], 1), sp(L[27], 1)
        fb, fs, fr = sp(L[48], 1), sp(L[50], 1), sp(L[65], 1)
        # tick = tm.strftime("%H%M%S.") + tm.strftime("%f")[:4]
        # tm = float(tm.strftime("%H%M%S.%f"))
        D = {'tick': tm, 'stk': stk, 'ceil': ceil, 'floor': floor, 'adj': adj, 'b3': b3, 'nsb3': nsb3, 'b2': b2, 'nsb2': nsb2, \
             'b1': b1, 'nsb1': nsb1, 'trdg': trdg, 'nstr': nstr, 'diff': diff, 'chng': chng, 's1': s1, 'nss1': nss1, \
             's2': s2, 'nss2': nss2, 's3': s3, 'nss3': nss3, 'ns': ns, 'high': high, 'low': low, 'fb': fb, 'fs': fs, 'fr': fr}
        return D
    else:
        return None
        # path = f"{home}/data/intraday/raw/{date}.npy"
        # arr = None if not os.path.exists(path) else load_file(path)
        # arr = vstack_row(arr, row)
        # save_file(arr, path)

# async def crawl(STK):
#     uri = "wss://iboard-pushstream.ssi.com.vn/realtime"
#     date, tm = get_date_time()
#     while True:
#         count, flag = 0, True
#         async with websockets.connect(uri) as WS:
#             tm = datetime.now()
#             tm = tm.strftime("%H:%M:%S")
#             print(f"Connected to server at {tm}.")
#             para = {"type": "sub", "topic": "stockRealtimeByListV2", "component": "priceTableEquities", "topic": "stockRealtimeByListV2", \
#                "type": "sub", "variables": STK}
#             while count <= 1:
#                 try:
#                     await WS.send(json.dumps(para))
#                     count += 1
#                     while flag == True:
#                         try:
#                             message = await WS.recv()
#                             read_message(message, date)
#                         except Exception as e:
#                             flag = False
#                             print(f"Error:\n{e}")
#                 except:
#                     count = 2

In [9]:
async def crawl(STK):
    uri = "wss://iboard-pushstream.ssi.com.vn/realtime"
    date, tm = get_date_time()
    while True:
        count, nmax, flag, df, buffer, err, nrow = 0, 10, True, None, [], '', 0
        COL = ['tick', 'stk', 'ceil', 'floor', 'adj', 'b3', 'nsb3', 'b2', 'nsb2', 'b1', 'nsb1', 'trdg', 'nstr', 'diff', 'chng', \
               's1', 'nss1', 's2', 'nss2', 's3', 'nss3', 'ns', 'high', 'low', 'fb', 'fs', 'fr']
        if os.path.exists(f"{home}/data/intraday/raw/{date}.fea"):
            df = load_file(f"{home}/data/intraday/raw/{date}.fea")
        else:
            df = pd.DataFrame(columns=COL)
            df.columns = df.columns.map(str)
        async with websockets.connect(uri) as WS:
            para = {"type": "sub", "topic": "stockRealtimeByListV2", "component": "priceTableEquities", "topic": "stockRealtimeByListV2", \
               "type": "sub", "variables": STK}
            while count < nmax:
                try:
                    await WS.send(json.dumps(para))
                    count += 1
                    while flag == True:
                        try:
                            tm = datetime.now()
                            tm = tm.strftime("%H:%M:%S")
                            clear_output(wait=True)
                            print(f"\ttime = {tm}  |  nrow = {nrow}.\n Previous errors:\n\n{err}")
                            message = await WS.recv()
                            row = read_message(message, date)
                            if row is not None:
                                buffer.append(row)
                            if len(buffer) >= 20:
                                df_new = pd.DataFrame(buffer)
                                if not df_new.empty and not df_new.isna().all(axis=None):
                                    df = pd.concat([df, df_new], ignore_index=True)
                                nrow = len(df.index)
                                save_file(df, f"{home}/data/intraday/raw/{date}.fea")
                                buffer = []
                        except Exception as err:
                            flag = False
                            print(err)
                except:
                    count = nmax

In [ ]:
# main
STK = collect_stk()
await crawl(STK)

	time = 14:46:57  |  nrow = 704660.
 Previous errors:


